<a href="https://colab.research.google.com/github/Anastasiia-hub0/unsolved-homicides-analysis/blob/main/unsolved_homicides_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unsolved Homicides: Which Case Characteristics Are Associated with a Case Staying Unsolved?
**Module:** Data Analytics (DBMS71008)- CA2  
**Student:** Anastasiia Tkachenko  
**Dataset:** Murder Accountability Project (MAP) – case-level Supplementary Homicide Report  
**GitHub:** https://github.com/Anastasiia-hub0/unsolved-homicides-analysis

# 1. Business Understanding

## 1.1 Context and problem
In the USA, many murder cases are never solved. Police do not have unlimited investigators and time, so they have to choose which cases need more attention.

In this project I want to find out which things we know about a case at the start (for example place, year, weapon, age and sex of the victim) are connected with the case staying unsolved.

**Research question:** Which characteristics of a homicide case that are known when it is reported are associated with the case staying unsolved, and can we use them to predict if the case will be solved?

**Unit of analysis:** one row = one murder case.

## 1.2 Stakeholders and users
- **Who gives the data:** the FBI reports, collected by the Murder Accountability Project (MAP).
- **Who uses the result:** the police and the people who decide how to share resources between murder cases. They decide which cases need more investigators and more time.
- **Who is affected:** the families and friends of the victims. They will not use the model, but the result matters to them. They want the offender to be found and punished by law. Also people in areas where cases are solved less often.

## 1.3 Business objective
Help to decide which murder cases need extra attention, so fewer cases stay unsolved.

## 1.4 Analytics objective
This is a binary classification task. The model predicts if a case is solved or unsolved, using only information known when the case is reported.

- **Target:** solved or unsolved.
- **Positive class = unsolved = 1.** Positive does not mean something good. It means the thing we are looking for. We look for unsolved cases, because these are the cases that need attention. Solved cases are already solved, so they are class 0.

## 1.5 Success criteria
**Technical**
- The model should be better than the baseline (the baseline always says "solved") in recall and F1 for unsolved cases. The baseline gets zero in both.
- The main metric is recall for the unsolved class. I also check precision and F1, so the list of risky cases is not too big and useless.
- I do not use accuracy alone. If most cases are solved, a model that always says "solved" has high accuracy but finds no unsolved cases.

**Practical**
- The list of risky cases should be easy to explain, so we can say which factors make a case more likely to stay unsolved.
- The model only helps. A human makes the final decision.

## 1.6 Which mistake is worse
- **Worse mistake:** the model says the case will be solved, but it stays unsolved. This case will not get extra resources, and the victim's family gets no justice.
- **Smaller mistake:** the model says the case will stay unsolved, but it would have been solved. We just spend a bit of extra attention.

So missing an unsolved case is worse than a false alarm. That is why recall is the most important metric here.

## 1.7 Constraints and assumptions
- **Time:** the project is due on 4 December 2026.
- **Old data:** the data is historical, so it may not show how police work now.
- **Not complete:** not every murder is reported to the FBI.
- **Data leakage:** columns about the offender (age, sex, relationship) are probably filled only for solved cases. We do not know them when a case starts, so I should not use them as inputs. I will check this in Data Understanding.
- **Missing values** can be written as "Unknown" or 999, not as empty cells.
- **Class imbalance** is possible. I will check it.
- **Ethics:** the data has victim race and location, so I need to talk about bias and fairness. The results show a connection, not a cause.
- **Tools:** Google Colab, Python and GitHub.
- **Licence:** I need to write the source and licence of the data.

## 1.8 Initial project plan
I will follow the six CRISP-DM phases:
1. **Business Understanding** – this section.
2. **Data Understanding** – source, what one row means, variables, data quality (missing values, "Unknown", leakage), balance of solved and unsolved.
3. **Data Preparation** – choose columns, clean, encode. I will keep a preparation log.
4. **Modelling** – baseline first, then logistic regression. Train/test split with a fixed random seed.
5. **Evaluation** – confusion matrix, precision, recall, F1, mistakes of the model, bias and ethics.
6. **Deployment** – how the police could use a list of risky cases, what should not be automated, future work.

# 2. Data Understanding

## 2.1 Initial Data Collection Report

**Data source:** The data comes from the Murder Accountability Project (MAP), a non-profit organisation that collects homicide data in the USA. I downloaded the file from their website (murderdata.org, page "Data & Docs"). The file is called `SHR76_25a.csv`.

**Date of access:** I downloaded the file on 17 October 2026.

**Licence:** I could not find any licence or terms of use on the MAP website. The page only says "Copyright 2019 by Murder Accountability Project", and that the files are available to police and the general public. I will name MAP as the source in my references. I will not upload the dataset to GitHub, I will only explain how to download it.

**How the data was collected:** The main part is the FBI Supplementary Homicide Report (SHR), which police agencies in the USA send to the FBI. MAP also added nearly 39,000 homicides that it got through the Freedom of Information Act, because they were not reported to the FBI.

**Population, time and place:** Homicide cases in the USA, from 1976 to 2025.

**Size and files:** One file with 929,433 rows and 31 columns. There is only one table, so nothing needs to be joined.

**Known limits:** On the MAP website it is written that nearly 39,000 homicides were not reported to the Justice Department, and MAP got them with the Freedom of Information Act. This means the FBI reports alone are not complete, so some cases can be missing even in this file. I also have to check how many values are "Unknown".

In [ ]:
import pandas as pd

df = pd.read_csv("SHR76_25a-2.csv", low_memory=False)
df.head()

,ID,CNTYFIPS,Ori,State,Agency,Agentype,Source,Solved,Year,Month,...,OffEthnic,Weapon,Relationship,Circumstance,Subcircum,VicCount,OffCount,FileDate,MSA,filter_$
0,197609001AL00400,"Autauga, AL",AL00400,Alabama,Autauga County,Sheriff,FBI,No,1976,September,...,Unknown or not reported,Other or type unknown,Relationship not determined,Other,NaN,0,0,30180.0,"Montgomery, AL",Not Selected
1,197701001AL00400,"Autauga, AL",AL00400,Alabama,Autauga County,Sheriff,FBI,Yes,1977,January,...,Unknown or not reported,Other or type unknown,Acquaintance,Brawl due to influence of alcohol,NaN,0,0,30180.0,"Montgomery, AL",Not Selected
2,197703001AL00400,"Autauga, AL",AL00400,Alabama,Autauga County,Sheriff,FBI,Yes,1977,March,...,Unknown or not reported,"Handgun - pistol, revolver, etc",Acquaintance,Other arguments,NaN,0,1,30180.0,"Montgomery, AL",Not Selected
3,197703001AL00401,"Autauga, AL",AL00401,Alabama,Prattville,Municipal police,FBI,Yes,1977,March,...,Unknown or not reported,Shotgun,Husband,Other arguments,NaN,0,0,30180.0,"Montgomery, AL",Not Selected
4,197708001AL00400,"Autauga, AL",AL00400,Alabama,Autauga County,Sheriff,FBI,Yes,1977,August,...,Unknown or not reported,Knife or cutting instrument,Acquaintance,Circumstances undetermined,NaN,0,0,30180.0,"Montgomery, AL",Not Selected


In [ ]:
df.shape

(929433, 31)

**What this shows:** The dataset has 929,433 rows and 31 columns. One row is one homicide case, so the unit of analysis is a case. This is a lot of data, enough for a train/test split and a model. The data covers the years 1976–2025.

In [ ]:
df.columns

Index(['ID', 'CNTYFIPS', 'Ori', 'State', 'Agency', 'Agentype', 'Source',
       'Solved', 'Year', 'Month', 'Incident', 'ActionType', 'Homicide',
       'Situation', 'VicAge', 'VicSex', 'VicRace', 'VicEthnic', 'OffAge',
       'OffSex', 'OffRace', 'OffEthnic', 'Weapon', 'Relationship',
       'Circumstance', 'Subcircum', 'VicCount', 'OffCount', 'FileDate', 'MSA',
       'filter_$'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 929433 entries, 0 to 929432
Data columns (total 31 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   ID            929433 non-null  object 
 1   CNTYFIPS      929433 non-null  object 
 2   Ori           929433 non-null  object 
 3   State         929433 non-null  object 
 4   Agency        929433 non-null  object 
 5   Agentype      929433 non-null  object 
 6   Source        929433 non-null  object 
 7   Solved        929433 non-null  object 
 8   Year          929433 non-null  int64  
 9   Month         929433 non-null  object 
 10  Incident      929433 non-null  int64  
 11  ActionType    929433 non-null  object 
 12  Homicide      929433 non-null  object 
 13  Situation     929433 non-null  object 
 14  VicAge        929433 non-null  int64  
 15  VicSex        929433 non-null  object 
 16  VicRace       929433 non-null  object 
 17  VicEthnic     929433 non-null  object 
 18  OffA

**What this shows:** The table has 31 columns: 24 are text and 7 are numbers. The target is `Solved`.

Only two columns have really empty values: `Subcircum` (almost all empty) and `FileDate` (about 24,900 empty). Other columns look full, but unknown values may be written as "Unknown" or 999, so I need to check this.

Columns about the offender (`OffAge`, `OffSex`, `OffRace`, `Relationship` and others) are known only after the offender is found. They may cause data leakage, so I will check them before choosing features.